In [2]:
import os
import re
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# =========================================================================
# 1. THE BRAIN: Dynamic Phase-Weighted Loss (DPW-Loss)
# =========================================================================
class DPWLoss(nn.Module):
    def __init__(self, total_epochs):
        super().__init__()
        self.total_epochs = total_epochs
        self.ce = nn.CrossEntropyLoss()
        
    def focal_loss(self, inputs, targets, alpha=0.25, gamma=2.0):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = alpha * (1 - pt) ** gamma * ce_loss
        return focal_loss.mean()

    def dice_loss(self, inputs, targets, smooth=1.0):
        # Convert targets to one-hot for Dice calculation
        preds = F.softmax(inputs, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=inputs.shape[1]).permute(0, 3, 1, 2).float()
        
        intersection = (preds * targets_one_hot).sum(dim=(2, 3))
        union = preds.sum(dim=(2, 3)) + targets_one_hot.sum(dim=(2, 3))
        dice = 1 - (2. * intersection + smooth) / (union + smooth)
        return dice.mean()

    def forward(self, inputs, targets, epoch):
        # Calculate individual losses
        l_ce = self.ce(inputs, targets)
        l_focal = self.focal_loss(inputs, targets)
        l_dice = self.dice_loss(inputs, targets)

        # Dynamic Phase Weighting based on epoch progress
        progress = epoch / self.total_epochs
        
        if progress < 0.33:
            # Phase 1: Foundational Learning (CE dominant)
            a, b, g = 1.0, 0.1, 0.1
        elif progress < 0.66:
            # Phase 2: Addressing Imbalance (Focal dominant)
            a, b, g = 0.1, 1.0, 0.1
        else:
            # Phase 3: Boundary Refinement (Dice dominant)
            a, b, g = 0.1, 0.5, 1.0

        return (a * l_ce) + (b * l_focal) + (g * l_dice)

# =========================================================================
# 2. THE BODY: RepMobileunit (Edge-Optimized Backbone Block)
# =========================================================================
class RepMobileunit(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.deploy = False
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride

        # Training-time multi-branch topology
        # Branch 1: 3x3 Depthwise
        self.dw_3x3 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels)
        )
        # Branch 2: 1x1 Depthwise
        self.dw_1x1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 1, stride=stride, padding=0, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels)
        )
        # Branch 3: Identity (only if input/output match and stride is 1)
        self.identity = nn.BatchNorm2d(in_channels) if stride == 1 else None

        # Pointwise stage
        self.pw = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        if self.deploy:
            # Inference mode: completely streamlined (implemented via self.reparameterize())
            return self.relu(self.pw(self.reparameterized_dw(x)))

        # Training mode: parallel branches
        dw_out = self.dw_3x3(x) + self.dw_1x1(x)
        if self.identity:
            dw_out += self.identity(x)
        
        return self.relu(self.pw(dw_out))
    
    def reparameterize(self):
        """
        Call this after training to mathematically fuse dw_3x3, dw_1x1, and identity 
        into a single self.reparameterized_dw layer, dropping parameters by ~80%.
        (Placeholder for the matrix algebra fusion logic).
        """
        self.deploy = True

# =========================================================================
# 3. THE NECK: HBAA Lite (Heterogeneous Feature Aggregation)
# =========================================================================
class StripPooling(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        h_str = self.pool_h(x)
        w_str = self.pool_w(x)
        # Expand and fuse
        out = h_str.expand_as(x) + w_str.expand_as(x)
        return torch.sigmoid(self.conv(out)) * x

class HBAALite(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # Simulates Window Attention efficiently for edge devices
        self.local_window = nn.Conv2d(channels, channels, 4, padding=1, groups=channels)
        # Captures elongated lesions
        self.strip_pool = StripPooling(channels)
        self.fusion = nn.Conv2d(channels * 2, channels, 1)

    def forward(self, x):
        local_feat = self.local_window(x)
        strip_feat = self.strip_pool(x)
        concat = torch.cat([local_feat, strip_feat], dim=1)
        return self.fusion(concat) + x

# =========================================================================
# 4. THE FULL ARCHITECTURE
# =========================================================================
class CropDiseaseNet(nn.Module):
    def __init__(self, num_seg_classes=2, num_disease_classes=10):
        super().__init__()
        
        # Simple Encoder built from RepMobileunits
        self.enc1 = RepMobileunit(3, 32, stride=2)   # 112x112
        self.enc2 = RepMobileunit(32, 64, stride=2)  # 56x56
        self.enc3 = RepMobileunit(64, 128, stride=2) # 28x28
        
        # Neck
        self.hbaa = HBAALite(128)
        
        # Decoder (Upsampling back to 224x224)
        self.dec = nn.Sequential(
            nn.Upsample(scale_factor=8, mode='bilinear', align_corners=False),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, num_seg_classes, 1)
        )
        
        # Classification Head (Global Avg Pool -> Linear)
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, num_disease_classes)
        )

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        
        enhanced_feat = self.hbaa(x)
        
        seg_out = self.dec(enhanced_feat)
        cls_out = self.cls_head(enhanced_feat)
        
        return seg_out, cls_out

# =========================================================================
# 5. DATASET & TRAINING LOOP UPDATES
# =========================================================================
# (Include your BinaryPlantSegDataset and train_transform here verbatim)

def train_hybrid_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Your Kaggle Paths
    img_folder = "/kaggle/input/datasets/weitianqi/plantseg/plantsegv2/images/train/"
    mask_folder = "/kaggle/input/datasets/weitianqi/plantseg/plantsegv2/annotations/train/"

    # 1. Instantiate the dataset
    dataset = BinaryPlantSegDataset(
        img_dir=img_folder, 
        mask_dir=mask_folder, 
        max_samples=1000,          # Subsampled to avoid Kaggle timeouts
        transform=train_transform
    )
    
    # 2. Create the DataLoader
    loader = DataLoader(
        dataset,
        batch_size=32,             # Lower to 16 if Kaggle throws a CUDA Out of Memory error
        shuffle=True,
        num_workers=2,             # Keep at 2 to avoid Kaggle CPU RAM crashes
        pin_memory=True
    )
    
    # 3. Dynamically grab the number of classes
    num_disease_classes = len(dataset.label_to_idx)
    epochs = 10

    model = CropDiseaseNet(
        num_seg_classes=2,
        num_disease_classes=num_disease_classes
    ).to(device)

    # Instantiate the new Dynamic Phase-Weighted Loss for segmentation
    seg_criterion = DPWLoss(total_epochs=epochs)
    
    # Standard CE is fine for the image-level classification task
    cls_criterion = nn.CrossEntropyLoss() 
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    model.train()
    print("\nTraining Hybrid RepMobile + DPW-Loss Model...")
    
    for epoch in range(epochs):
        running_seg_loss = 0.0
        running_cls_loss = 0.0

        for images, masks, labels in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if device.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    seg_out, cls_out = model(images)
                    
                    # NOTE: Pass 'epoch' into the segmentation criterion!
                    loss_seg = seg_criterion(seg_out, masks, epoch)
                    loss_cls = cls_criterion(cls_out, labels)
                    total_loss = loss_seg + loss_cls

                scaler.scale(total_loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                seg_out, cls_out = model(images)
                loss_seg = seg_criterion(seg_out, masks, epoch)
                loss_cls = cls_criterion(cls_out, labels)
                total_loss = loss_seg + loss_cls
                total_loss.backward()
                optimizer.step()

            running_seg_loss += loss_seg.item()
            running_cls_loss += loss_cls.item()

        avg_seg = running_seg_loss / len(loader)
        avg_cls = running_cls_loss / len(loader)
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] - Seg Loss: {avg_seg:.4f} | Cls Loss: {avg_cls:.4f} | Total: {avg_seg + avg_cls:.4f}")

    torch.save(model.state_dict(), "hybrid_edge_disease_model.pth")
    print("\nSaved weights. Ready for reparameterization and edge deployment!")

In [ ]:
# Instantiate the dataset using the albumentations transforms defined earlier
    dataset = BinaryPlantSegDataset(
        img_dir=img_folder, 
        mask_dir=mask_folder, 
        max_samples=1000,          # Adjust this based on your GPU limits
        transform=train_transform  # Ensure this points to the train_transform A.Compose block
    )
    
    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    
    num_disease_classes = len(dataset.label_to_idx)